# NERVE Sensor Setup

This walkthrough introduces the five sensing streams in the NERVE dataset:

| Sensor | Role |
|--------|------|
| **DAVIS346** | Event (DVS) + optional APS frames from iniVation DAVIS346 |
| **EVK4 (Prophesee)** | High-resolution event stream from Prophesee Metavision EVK4 |
| **Infineon Position2Go** | FMCW radar (24 GHz), co-recorded with DAVIS346 in the `.rad` file |
| **TI AWR1443BOOST** | FMCW mmWave radar (77 GHz, Texas Instruments), separate recording |
| **Intel L515 RGB** | Color camera aligned with depth |
| **Intel L515 Depth** | Time-of-flight depth aligned with RGB |

Together they provide asynchronous visual, geometric, and radar measurements for multi-modal fusion. Included in this repository is also a 3D model (`NERVE_setup.blend`) of the sensor setup which allows for the recreation of geometric relations.

![NERVE Sensor Setup 3D Model](figures/NERVE_setup_3DModel.png)

First we download an example session based on size to illustrate the raw data of each sensor further.



In [ ]:
from nerve import config, remote
from nerve.registry import all_sessions
from importlib.resources import files
from pathlib import Path
import json

sessions = all_sessions()
smallest = min(sessions, key=lambda s: s.size_bytes)
print(f"Smallest session: {smallest.name}  "
      f"({smallest.size_bytes / 1e9:.2f} GB archive, split={smallest.split})")

session_dir = remote.download_session(smallest.name)
SESSION = Path(session_dir)
print(f"Session path: {SESSION}")


## Sensor specifications

| Sensor | Resolution / output | Approx. frame / event rate |
|--------|---------------------|------------------------------|
| DAVIS346 | 346×260 px (DVS) | kHz-scale events; APS up to ~50 Hz | 
| EVK4 | 1280×720 (HD mode) | Very high event throughput | 
| TI AWR1443BOOST | Range-Doppler cubes | Chirp / frame rate set per capture | 
| L515 RGB | 1280×720 | ~60 Hz video |
| L515 Depth | Matched to RGB | ~30 Hz depth video |


## Raw sensor data for the example session

Each downloaded session contains raw data for every sensor. Below we open the session downloaded above (the smallest by archive size) and inspect the on-disk layout, array shapes, and basic statistics for each modality.

In [2]:
import os, h5py, cv2

# ── Session metadata ────────────────────────────────────────────────
meta = json.loads((SESSION / "session_metadata.json").read_text())
system = meta["sml:System"]
tp = system["sml:validTime"]["gml:TimePeriod"]
print(f"Session : {system['gml:name']}")
print(f"Duration: {tp['duration_formatted']}  ({tp['duration_seconds']:.1f} s)")
print(f"Start   : {tp['gml:beginPosition']}")

sensors = meta["swe:DataRecord"]["sensors_available"]
print(f"Sensors : {', '.join(k for k, v in sensors.items() if v)}")

# ── Timing offsets ──────────────────────────────────────────────────
timings = json.loads((SESSION / "timings.json").read_text())
ref = min(timings.values())
print(f"\nTiming offsets (seconds from earliest sensor):")
for name, ts in sorted(timings.items(), key=lambda x: x[1]):
    print(f"  {name:>12s}: +{ts - ref:.6f} s")

# ── DAVIS346 events (HDF5) ──────────────────────────────────────────
print("\n" + "=" * 60)
print("DAVIS346  --  davis/events.hdf5")
print("=" * 60)
with h5py.File(SESSION / "davis" / "events.hdf5", "r") as f:
    ev = f["events"]
    print(f"  Sensor name   : {ev.attrs['name']}")
    print(f"  Resolution    : {ev.attrs['width']} × {ev.attrs['height']} px")
    print(f"  Total events  : {ev.attrs['total_events']:,}")
    print(f"  Duration      : {ev.attrs['total_time_uS'] / 1e6:.2f} s")
    print(f"  Avg event rate: {ev.attrs['total_events'] / (ev.attrs['total_time_uS'] / 1e6):,.0f} ev/s")
    print(f"  HDF5 dtype    : {ev.dtype}")
    print(f"  Fields        : {ev.dtype.names}")
    sample = ev[:5]
    print(f"  First 5 events:\n    {'t':>12s} {'x':>5s} {'y':>5s} {'p':>2s}")
    for e in sample:
        print(f"    {int(e['t']):12d} {int(e['x']):5d} {int(e['y']):5d} {int(e['p']):2d}")
    if "support_indexes" in f:
        si = f["support_indexes"]
        print(f"  Support index : {si.shape[0]:,} entries  "
              f"(timestep {si.attrs['timestep_uS']} µs)")

# ── EVK4 / Prophesee events (HDF5) ─────────────────────────────────
print("\n" + "=" * 60)
print("EVK4 (Prophesee)  --  prophesee/events.hdf5")
print("=" * 60)
with h5py.File(SESSION / "prophesee" / "events.hdf5", "r") as f:
    ev = f["events"]
    print(f"  Sensor name   : {ev.attrs['name']}")
    print(f"  Resolution    : {ev.attrs['width']} × {ev.attrs['height']} px")
    print(f"  Total events  : {ev.attrs['total_events']:,}")
    print(f"  Duration      : {ev.attrs['total_time_uS'] / 1e6:.2f} s")
    print(f"  Avg event rate: {ev.attrs['total_events'] / (ev.attrs['total_time_uS'] / 1e6):,.0f} ev/s")
    print(f"  HDF5 dtype    : {ev.dtype}")
    print(f"  Fields        : {ev.dtype.names}")
    sample = ev[:5]
    field_order = ev.dtype.names
    print(f"  First 5 events:\n    {'t':>12s} {'x':>5s} {'y':>5s} {'p':>2s}")
    for e in sample:
        print(f"    {int(e['t']):12d} {int(e['x']):5d} {int(e['y']):5d} {int(e['p']):2d}")

# ── Position2Go + DAVIS346 (.rad binary) ───────────────────────────
print("\n" + "=" * 60)
print("Position2Go + DAVIS346  --  radar_and_davis346_events.rad")
print("=" * 60)
rad_path = SESSION / "radar_and_davis346_events.rad"
rad_size = rad_path.stat().st_size
print(f"  File size: {rad_size / (1024**3):.2f} GB")
with open(rad_path, "rb") as f:
    magic = f.read(4)
    print(f"  Magic     : {magic}")
print("  (Binary format containing interleaved 24 GHz radar chirps and DVS events)")

# ── TI AWR1443BOOST radar (HDF5) ───────────────────────────────────
print("\n" + "=" * 60)
print("TI AWR1443BOOST  --  ti_radar/captured_data/set000/data.h5")
print("=" * 60)
radar_h5 = SESSION / "ti_radar" / "captured_data" / "set000" / "data.h5"
with h5py.File(radar_h5, "r") as f:
    ds = f["radar/dataset_1/data"]
    ts = f["radar/dataset_1/timestamps"]
    print(f"  ADC data shape : {ds.shape}  →  "
          f"(frames, chirp_loops, TX, RX, ADC_samples, I/Q)")
    print(f"  ADC dtype      : {ds.dtype}")
    print(f"  Num frames     : {ds.shape[0]:,}")
    duration = (ts[-1] - ts[0])
    print(f"  Duration       : {duration:.1f} s")
    print(f"  Frame rate     : {ds.shape[0] / duration:.2f} Hz")
    print(f"  Timestamps     : [{ts[0]:.3f} … {ts[-1]:.3f}] (epoch seconds)")
    print(f"  Chirp loops    : {ds.shape[1]}")
    print(f"  TX × RX        : {ds.shape[2]} × {ds.shape[3]}")
    print(f"  ADC samples    : {ds.shape[4]}  (I/Q pairs)")

# ── L515 Depth video (.mp4, 16-bit gray) ───────────────────────────
print("\n" + "=" * 60)
print("Intel L515 Depth  --  L515_depth.mp4")
print("=" * 60)
cap = cv2.VideoCapture(str(SESSION / "L515_depth.mp4"))
print(f"  Resolution   : {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))} × "
      f"{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
print(f"  Frame rate   : {cap.get(cv2.CAP_PROP_FPS):.1f} fps")
print(f"  Total frames : {int(cap.get(cv2.CAP_PROP_FRAME_COUNT)):,}")
depth_unit = float((SESSION / "L515_depth_unit.txt").read_text().strip())
print(f"  Depth unit   : {depth_unit} m/count")
print(f"  Encoding     : gray16le  (raw 16-bit depth, multiply by unit for meters)")
cap.release()

cap = cv2.VideoCapture(str(SESSION / "L515_depth_confidence.mp4"))
print(f"\n  Confidence video:")
print(f"    Resolution   : {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))} × "
      f"{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
print(f"    Total frames : {int(cap.get(cv2.CAP_PROP_FRAME_COUNT)):,}")
cap.release()

# ── DAVIS346 annotation videos ──────────────────────────────────────
print("\n" + "=" * 60)
print("DAVIS346 derived videos  --  davis/*.mp4")
print("=" * 60)
for name in sorted(os.listdir(SESSION / "davis")):
    if name.endswith(".mp4"):
        cap = cv2.VideoCapture(str(SESSION / "davis" / name))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f"  {name:25s}  {w}×{h}  {fps:.0f} fps  {n:,} frames")
        cap.release()

# ── COCO annotations ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("Annotations  --  COCO JSON format")
print("=" * 60)
for pov in ["davis", "prophesee", "rgb"]:
    ann_path = SESSION / pov / "annotations" / "annotations.json"
    if ann_path.exists():
        ann = json.loads(ann_path.read_text())
        cats_present = set()
        for a in ann["annotations"]:
            cat_id = a["category_id"]
            for c in ann["categories"]:
                if c["id"] == cat_id:
                    cats_present.add(c["name"])
        print(f"\n  {pov}/annotations/annotations.json:")
        print(f"    Images      : {len(ann['images']):,}")
        print(f"    Annotations : {len(ann['annotations']):,}")
        print(f"    Categories  : {len(ann['categories'])}")
        print(f"    Present     : {', '.join(sorted(cats_present))}")
        img0 = ann["images"][0]
        print(f"    First image : id={img0['id']}, {img0['width']}×{img0['height']}, "
              f"time={img0.get('time_ms', '?')} ms")
        a0 = ann["annotations"][0]
        print(f"    Sample ann  : bbox={[round(x,1) for x in a0['bbox']]}, "
              f"distance={a0.get('avg_distance', 'N/A')}")

=== rgb_to_davis.json ===

--- src (rgb) ---
  resolution: 1280 x 720
  intrinsics: {
  "fx": 905.030639648,
  "fy": 905.665710449,
  "cx": 653.86416574,
  "cy": 349.516906738,
  "skew": 0
}
  distortions: {
  "k1": 0.1414,
  "k2": 0.4705,
  "k3": 0.4217,
  "p1": 0.0017,
  "p2": 0.002
}

--- dst (davis) ---
  resolution: 346 x 260
  intrinsics: {
  "fx": 465.4087,
  "fy": 465.2258,
  "cx": 181.0787,
  "cy": 129.7596,
  "skew": -0.0751
}
  distortions: {
  "k1": -0.0551,
  "k2": 0.0168,
  "k3": 1.1436,
  "p1": -0.0001646,
  "p2": 0.0023
}

--- extrinsic src_to_dst ---
  rotation (3x3): [[0.9994000196456909, -0.00570000009611249, -0.033799998462200165], [0.0038999998942017555, 0.9984999895095825, -0.054499998688697815], [0.03400000184774399, 0.05429999902844429, 0.9979000091552734]]
  translation (3,): [-0.02825699932873249, 0.028812099248170853, 0.041074201464653015]


## Coordinate systems and extrinsic calibration

- **Intrinsics** map pixel coordinates to normalized rays in each camera’s optical frame.
- **Distortion** (Brown-Conrady or fisheye where applicable) is applied before undistortion / projection steps.
- **Extrinsics** `src_to_dst` describe the rigid transform from the source sensor frame (e.g. RGB) to the destination frame (e.g. DAVIS): rotation `rot` and translation `trans`. Combined with depth (or a plane hypothesis), labels can be **reprojected** between views using the bundled mapping JSON files.

NERVE ships precomputed mappings so you can align boxes, keypoints, and segmentations across modalities without re-running calibration.

In [3]:
from importlib.resources import files
import json

mapping_names = [
    "rgb_to_davis.json",
    "rgb_to_prophesee.json",
    "ti_radar_to_davis.json",
    "ti_radar_to_prophesee.json",
    "ti_radar_to_rgb.json",
]
base = files("nerve.data.mappings")

for name in mapping_names:
    p = base / name
    obj = json.loads(p.read_text(encoding="utf-8"))
    src = obj.get("src", {})
    dst = obj.get("dst", {})
    print(f"\n{name}")
    print(f"  src : {src.get('name', src)}")
    print(f"  dst : {dst.get('name', dst)}")
    if isinstance(src, dict) and "width" in src:
        print(f"  src resolution: {src['width']}x{src['height']}")
    if isinstance(dst, dict) and "width" in dst:
        print(f"  dst resolution: {dst['width']}x{dst['height']}")
    print(f"  has extrinsic src_to_dst: {'src_to_dst' in obj}")


rgb_to_davis.json
  src : rgb
  dst : davis
  src resolution: 1280x720
  dst resolution: 346x260
  has extrinsic src_to_dst: True

rgb_to_prophesee.json
  src : rgb
  dst : prophesee
  src resolution: 1280x720
  dst resolution: 1280x720
  has extrinsic src_to_dst: True

ti_radar_to_davis.json
  src : ti_radar
  dst : davis
  dst resolution: 346x260
  has extrinsic src_to_dst: True

ti_radar_to_prophesee.json
  src : ti_radar
  dst : prophesee
  dst resolution: 1280x720
  has extrinsic src_to_dst: True

ti_radar_to_rgb.json
  src : ti_radar
  dst : rgb
  dst resolution: 1280x720
  has extrinsic src_to_dst: True
